# Imports

In [ ]:
import json
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.multidate import load_date
from uavloc.datasets import dataset_ref
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import SegmentationFeatures, label_probabilities, load_segmenter
from uavloc.geo import footprint_m
from uavloc.matching.correlator import CAMERA_ASPECT_W_OVER_H, PositionCorrelator
from uavloc.matching.oracle import QueryCorrelator, gt_labels, gt_query
from uavloc.matching.benchmark import HARM_MARGIN_M, paired_frame_bootstrap, run_benchmark
from uavloc.track import REPO_ROOT, set_seed

# Params

In [ ]:
BUILD = "corrector-identity-bench"
MATCHINGS = ["weighted-ncc", "cosine"]   # per-class NCC vs one joint correlation over the whole map
MASK_SOURCE = "model"                    # "model": the segmenter on both sides | "gt": hand labels on both sides
LABEL_SOURCE = "annotated"               # hand-corrected labels; OSM lacks woody (006)

WINDOW = "rect"                          # 006.1 setup, ungated, so only the matching mode differs
SEARCH_SCALE = 1.7
PLACES = ["bushcha-50p2422-25p8305", "birky-49p9105-23p9192"]
MAP_DATE = "newest"
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 10.0
N_DIRECTIONS = 8
SEED = 42
DEVICE = "mps"
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0
SEG_ARCHITECTURE = "lraspp"
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
SEG_CHANNELS = "all"
MATCH_GSD_M = 0.385
FRAMES_DATASET = "village-crossdate-frames"
N_BOOTSTRAP = 10_000
CI_PCT = 95

# Data

In [ ]:
set_seed(SEED)
OUT = REPO_ROOT / "builds" / BUILD


def run_config(matching: str) -> dict:
    return {"method": "segmentation", "matching": matching, "class_weights": CLASS_WEIGHTS,
            "noise_m": NOISE_M, "primary_noise_m": PRIMARY_NOISE_M, "n_directions": N_DIRECTIONS,
            "window": WINDOW, "search_scale": SEARCH_SCALE, "search_radius_m": None, "prior_sigma_m": None,
            "gate": "none", "fov_w_deg": FOV_W_DEG,
            "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
            "seg_architecture": SEG_ARCHITECTURE, "seg_checkpoint_sat": SEG_CHECKPOINT,
            "seg_checkpoint_drone": SEG_CHECKPOINT, "seg_channels": SEG_CHANNELS,
            "drone_seg_gsd_m": MATCH_GSD_M, "match_gsd_m": MATCH_GSD_M,
            "frames_dataset": FRAMES_DATASET, "frames_dataset_ref": dataset_ref(FRAMES_DATASET),
            "places": PLACES, "map_date": MAP_DATE,
            "mask_source": MASK_SOURCE,
            "label_source": LABEL_SOURCE if MASK_SOURCE == "gt" else None}

In [ ]:
def frame_place(frame: DroneFrame) -> str:
    return json.loads((frame.image_path.parent / "meta.json").read_text())["place"]

In [ ]:
frames = [f for f in load_frames(FRAMES_DATASET) if frame_place(f) in PLACES]
places = {place: [f for f in frames if frame_place(f) == place] for place in PLACES}
maps = {place: load_date(place, MAP_DATE) for place in PLACES}
labels = {place: gt_labels(place, LABEL_SOURCE) for place in PLACES} if MASK_SOURCE == "gt" else {}
{place: len(members) for place, members in places.items()}

# Model

In [ ]:
model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, DEVICE, SEG_ARCHITECTURE)

In [ ]:
def place_features(place: str) -> SegmentationFeatures:
    """Satellite side: the segmenter on the map raster, or the hand labels as one-hot probabilities."""
    if MASK_SOURCE == "model":
        return SegmentationFeatures(model, model, maps[place], SEG_CHANNELS, MATCH_GSD_M, MATCH_GSD_M, DEVICE)
    return SegmentationFeatures.from_satellite_probabilities(
        model, label_probabilities(labels[place], MATCH_GSD_M), SEG_CHANNELS, MATCH_GSD_M, MATCH_GSD_M, DEVICE)

In [ ]:
features = {place: place_features(place) for place in PLACES}

In [ ]:
def query_map(frame: DroneFrame, place: str) -> np.ndarray:
    """Drone side as GT: hand labels cut heading-up at the frame's true pose, on the match grid."""
    return gt_query(labels[place], frame, footprint_m(frame.altitude_m, FOV_W_DEG, CAMERA_ASPECT_W_OVER_H), MATCH_GSD_M)

In [ ]:
def place_correlator(place: str, config) -> PositionCorrelator:
    """Both sides from the same source: the model on frame and map, or GT on both."""
    if MASK_SOURCE == "model":
        return PositionCorrelator(features[place], config)
    return QueryCorrelator(features[place], config, lambda frame, place=place: query_map(frame, place))

# Run

In [ ]:
def matching_records(matching: str) -> pd.DataFrame:
    """Every draw under one matching mode; the heatmap itself changes, so surfaces cannot be shared."""
    config = correlator_config(run_config(matching))
    parts = []
    for place in PLACES:
        correlator = place_correlator(place, config)
        parts += [run_benchmark(correlator, places[place], noise_m, N_DIRECTIONS, SEED).assign(place=place)
                  for noise_m in NOISE_M]
    records = pd.concat(parts, ignore_index=True).assign(matching=matching)
    records["harm"] = records["error_m"] > records["prior_error_m"] + HARM_MARGIN_M
    return records

In [ ]:
records = pd.concat([matching_records(m) for m in tqdm(MATCHINGS, desc="matching")], ignore_index=True)
records.to_csv(OUT / "runs" / f"matching-{MASK_SOURCE}-records.csv", index=False)
len(records)

In [ ]:
def summary(group: pd.DataFrame) -> pd.DataFrame:
    return group.groupby("noise_m").apply(lambda d: pd.Series({
        "mean_m": d["error_m"].mean(), "median_m": d["error_m"].median(),
        "p90_m": np.percentile(d["error_m"], 90), "max_m": d["error_m"].max(),
        "harm_share": d["harm"].mean(), "reachable_share": d["reachable"].mean(),
    }), include_groups=False).round(3)

In [ ]:
table = pd.concat({m: summary(g) for m, g in records.groupby("matching")}, names=["matching"])
table.to_csv(OUT / "runs" / f"matching-{MASK_SOURCE}-summary.csv")
table

In [ ]:
paired = {f"{n:g} m": paired_frame_bootstrap(records[records["matching"] == "cosine"],
                                             records[records["matching"] == "weighted-ncc"],
                                             n, N_BOOTSTRAP, CI_PCT, SEED) for n in NOISE_M}
paired = pd.DataFrame(paired).T.round(3)
paired.to_csv(OUT / "runs" / f"matching-{MASK_SOURCE}-paired.csv")
paired

In [ ]:
by_place = records.groupby(["matching", "place", "noise_m"])["error_m"].mean().unstack(0).round(3)
by_place.to_csv(OUT / "runs" / f"matching-{MASK_SOURCE}-by-place.csv")
by_place

# Plots

In [ ]:
def plot_cdf(records: pd.DataFrame, noise_m: float, ax: plt.Axes) -> None:
    for matching, group in records[records["noise_m"] == noise_m].groupby("matching"):
        errors = np.sort(group["error_m"].to_numpy())
        ax.plot(errors, np.linspace(0, 1, len(errors)), label=matching)
    ax.axvline(noise_m, color="grey", ls=":", lw=1)
    ax.set_xlabel("position error, m"); ax.set_title(f"{noise_m:g} m injected, {MASK_SOURCE} masks")
    ax.legend(fontsize=8); ax.grid(alpha=.3)

In [ ]:
fig, axes = plt.subplots(1, len(NOISE_M), figsize=(4 * len(NOISE_M), 3.6), sharey=True)
for ax, noise_m in zip(axes, NOISE_M):
    plot_cdf(records, noise_m, ax)
axes[0].set_ylabel("share of draws")
fig.tight_layout()
fig.savefig(OUT / "figures" / f"matching-{MASK_SOURCE}-cdf.png", dpi=150)